# RetinaScan AI — Grad-CAM Explainability
## Visual Interpretation of ConvNeXt-Tiny Predictions on OCT Retinal Scans

**Project:** RetinaScan AI (OCT-based Retinal Disease Classification with Explainable AI)
**Team:** Murali A, Niranjan T, Praveen K
**Guide:** Dr. C. Santhosh Kumar
**Sona College of Technology — Department of Information Technology**

---

### What this notebook does

This notebook loads the trained ConvNeXt-Tiny model (test accuracy 99.90%) and generates **Grad-CAM heatmaps** to visualize which retinal regions the model attended to when making each prediction. This is the interpretability branch of the RetinaScan AI system and answers the question:

> "The model correctly classifies OCT scans — but is it looking at clinically relevant regions?"

### Method: Grad-CAM (Gradient-weighted Class Activation Mapping)

1. Forward pass an OCT image through ConvNeXt-Tiny
2. Extract feature maps from the last convolutional stage
3. Compute the gradient of the predicted class score with respect to those feature maps
4. Weight-average the feature maps by the mean gradient (importance score per channel)
5. Apply ReLU and upsample to input resolution
6. Overlay the resulting heatmap on the original OCT image

Red/yellow regions in the heatmap indicate areas of high model attention.

### Outputs generated

- `gradcam_comparison.png` — 4×3 grid: 4 disease classes × 3 columns (original, heatmap, overlay)
- `gradcam_grid_full.png` — expanded 8-sample grid (2 samples per class)
- Individual PNG per sample for detailed inspection


## 1 · Setup and Model Loading

In [2]:
# Core imports
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import convnext_tiny
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os
from pathlib import Path

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


In [3]:
# Rebuild the same ConvNeXt-Tiny architecture used during training
import torch.nn as nn

MODEL_PATH = "/kaggle/working/final_results/best_convnext_tiny_oct2017.pth"
CLASSES = ["CNV", "DME", "DRUSEN", "NORMAL"]
NUM_CLASSES = 4

# Build architecture (no pretrained weights needed — we'll load our own)
model = convnext_tiny(weights=None)

# Replace the final classifier layer to match training-time setup
num_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(num_features, NUM_CLASSES)

# Load our trained checkpoint
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# Handle different checkpoint save formats
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded model_state_dict from checkpoint")
    if 'epoch' in checkpoint:
        print(f"  Trained for {checkpoint['epoch']} epochs")
    if 'val_acc' in checkpoint:
        print(f"  Best validation accuracy: {checkpoint['val_acc']:.2f}%")
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
    print(f"✓ Loaded state_dict from checkpoint")
else:
    # Assume it's a raw state dict
    model.load_state_dict(checkpoint)
    print(f"✓ Loaded raw state_dict")

model = model.to(device)
model.eval()
print(f"\n✓ ConvNeXt-Tiny loaded successfully on {device}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/final_results/best_convnext_tiny_oct2017.pth'

## 2 · Grad-CAM Implementation

We attach forward and backward hooks to the last convolutional stage of ConvNeXt-Tiny (`model.features[-1]` — the final ConvNeXt stage). During a forward pass, we capture the feature maps. During the backward pass on the predicted class score, we capture the gradients. The Grad-CAM heatmap is then the ReLU of the gradient-weighted sum of feature maps.


In [ ]:
class GradCAM:
    """Grad-CAM implementation for ConvNeXt-Tiny.
    Hooks the last convolutional stage to capture feature maps and gradients."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def __call__(self, input_tensor, class_idx=None):
        # Forward pass
        self.model.zero_grad()
        output = self.model(input_tensor)

        # Use predicted class if none specified
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Backward pass on target class score
        class_score = output[0, class_idx]
        class_score.backward(retain_graph=True)

        # Compute Grad-CAM
        # activations: [1, C, H, W]  |  gradients: [1, C, H, W]
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # [1, C, 1, 1]
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        cam = F.relu(cam)

        # Normalize to [0, 1]
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, F.softmax(output, dim=1)[0].detach().cpu().numpy()


# Attach Grad-CAM to the last ConvNeXt stage
# In torchvision's ConvNeXt-Tiny: model.features[-1] is the last stage
target_layer = model.features[-1]
gradcam = GradCAM(model, target_layer)
print(f"✓ Grad-CAM attached to layer: {type(target_layer).__name__}")

## 3 · Image Preprocessing and Overlay Helpers

In [ ]:
# Same transforms used during training
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def load_and_preprocess(image_path):
    """Load OCT image → return (raw_display_image, model_input_tensor)."""
    pil_img = Image.open(image_path).convert("L")  # grayscale
    # Raw image for display (resized to model input size, in [0, 255] uint8)
    display_img = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE)))
    # Model input tensor
    input_tensor = preprocess(pil_img).unsqueeze(0).to(device)
    return display_img, input_tensor

def overlay_heatmap(display_img_gray, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original grayscale OCT image."""
    # Resize heatmap to match display image
    heatmap_resized = cv2.resize(heatmap, (display_img_gray.shape[1], display_img_gray.shape[0]))
    # Apply JET colormap
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    # Convert display to RGB
    display_rgb = np.stack([display_img_gray]*3, axis=-1)
    # Weighted overlay
    overlay = cv2.addWeighted(display_rgb, 1 - alpha, heatmap_color, alpha, 0)
    return overlay, heatmap_color

print("✓ Preprocessing and overlay helpers ready")

## 4 · Select Sample Test Images

We pick **2 samples per class** from the official test set. These are images the model has never seen during training. This gives us 8 total samples for Grad-CAM analysis.


In [ ]:
# Locate the test set
import os
from pathlib import Path

# Search for the test directory (dataset folder name has a trailing space)
possible_test_dirs = [
    "/kaggle/input/kermany2018/OCT2017 /test",
    "/kaggle/input/kermany2018/OCT2017/test",
    "/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017 /test",
    "/kaggle/input/datasets/paultimothymooney/kermany2018/OCT2017/test",
]

TEST_DIR = None
for p in possible_test_dirs:
    if os.path.exists(p):
        TEST_DIR = Path(p)
        break

# Fallback — search
if TEST_DIR is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "CNV" in dirs and "DME" in dirs and "test" in root.lower():
            TEST_DIR = Path(root)
            break

assert TEST_DIR is not None, "❌ Could not locate test directory"
print(f"✓ Test directory: {TEST_DIR}")

# Pick 2 test images per class
SAMPLES_PER_CLASS = 2
samples = []  # list of (image_path, true_class_name, true_class_idx)

for class_idx, class_name in enumerate(CLASSES):
    class_dir = TEST_DIR / class_name
    files = sorted([f for f in class_dir.iterdir() if f.suffix.lower() in ('.jpeg', '.jpg', '.png')])
    for f in files[:SAMPLES_PER_CLASS]:
        samples.append((f, class_name, class_idx))

print(f"\n✓ Selected {len(samples)} test samples:")
for path, cls, idx in samples:
    print(f"  [{cls:<7}] {path.name}")

## 5 · Generate Grad-CAM for Each Sample

In [ ]:
# Run Grad-CAM on all samples
results = []  # list of dicts

for i, (img_path, true_class, true_idx) in enumerate(samples):
    display_img, input_tensor = load_and_preprocess(img_path)

    # Compute Grad-CAM using the model's predicted class
    heatmap, pred_idx, probs = gradcam(input_tensor)
    pred_class = CLASSES[pred_idx]
    confidence = probs[pred_idx] * 100

    # Build overlay
    overlay, heatmap_color = overlay_heatmap(display_img, heatmap, alpha=0.45)

    correct = (pred_idx == true_idx)
    results.append({
        'path': img_path,
        'true_class': true_class,
        'pred_class': pred_class,
        'confidence': confidence,
        'correct': correct,
        'display_img': display_img,
        'heatmap_color': heatmap_color,
        'overlay': overlay,
        'probs': probs,
    })

    tag = "✓" if correct else "✗"
    print(f"  {tag} [{i+1}/{len(samples)}] True: {true_class:<7} | Pred: {pred_class:<7} | Confidence: {confidence:5.2f}%")

correct_count = sum(1 for r in results if r['correct'])
print(f"\n✓ Grad-CAM complete: {correct_count}/{len(results)} samples correctly classified")

## 6 · Visualization — Comparison Grid

The main figure for the report and PPT. Each row shows one test sample with three views:
- **Original OCT scan** (what the model sees as input)
- **Grad-CAM heatmap** (which regions the model attended to)
- **Overlay** (heatmap superimposed on original, showing the attention region in retinal context)


In [ ]:
# Master figure — 8 rows × 3 columns (Original / Heatmap / Overlay)
n_rows = len(results)
fig, axes = plt.subplots(n_rows, 3, figsize=(11, 3 * n_rows))
fig.suptitle(
    "Grad-CAM Explainability Analysis — ConvNeXt-Tiny on OCT2017 Test Set",
    fontsize=14, fontweight='bold', y=1.001
)

for row, r in enumerate(results):
    # Column 0 — Original
    ax = axes[row, 0]
    ax.imshow(r['display_img'], cmap='gray')
    ax.set_title(f"Original OCT — True: {r['true_class']}", fontsize=10, fontweight='bold')
    ax.axis('off')

    # Column 1 — Heatmap alone
    ax = axes[row, 1]
    ax.imshow(r['heatmap_color'])
    ax.set_title("Grad-CAM Heatmap", fontsize=10, fontweight='bold')
    ax.axis('off')

    # Column 2 — Overlay with prediction result
    ax = axes[row, 2]
    ax.imshow(r['overlay'])
    color = 'green' if r['correct'] else 'red'
    symbol = '✓' if r['correct'] else '✗'
    ax.set_title(
        f"Overlay | Pred: {r['pred_class']} ({r['confidence']:.1f}%) {symbol}",
        fontsize=10, fontweight='bold', color=color
    )
    ax.axis('off')

plt.tight_layout()

# Save
OUTPUT_DIR = Path("/kaggle/working/final_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_path = OUTPUT_DIR / "gradcam_grid_full.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n✓ Saved to {save_path}")

## 7 · Compact 4-Class Comparison (for PPT Slide)

A 4-row grid showing one sample per class — cleaner for slide presentation than the 8-sample version.


In [ ]:
# Compact — one sample per class (the first of each)
first_per_class = []
seen_classes = set()
for r in results:
    if r['true_class'] not in seen_classes:
        first_per_class.append(r)
        seen_classes.add(r['true_class'])

fig, axes = plt.subplots(len(first_per_class), 3, figsize=(11, 3 * len(first_per_class)))
fig.suptitle(
    "Grad-CAM Visualizations — One Sample per Class",
    fontsize=14, fontweight='bold', y=1.001
)

for row, r in enumerate(first_per_class):
    axes[row, 0].imshow(r['display_img'], cmap='gray')
    axes[row, 0].set_title(f"Original OCT — {r['true_class']}", fontsize=11, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(r['heatmap_color'])
    axes[row, 1].set_title("Grad-CAM Heatmap", fontsize=11, fontweight='bold')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(r['overlay'])
    color = 'green' if r['correct'] else 'red'
    symbol = '✓' if r['correct'] else '✗'
    axes[row, 2].set_title(
        f"Prediction: {r['pred_class']} ({r['confidence']:.1f}%) {symbol}",
        fontsize=11, fontweight='bold', color=color
    )
    axes[row, 2].axis('off')

plt.tight_layout()
save_path = OUTPUT_DIR / "gradcam_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved to {save_path}")

## 8 · Individual Sample Analysis (Optional Deep-Dive)

Saves each sample as a standalone PNG for individual analysis or paper figures.


In [ ]:
# Save each sample as an individual PNG
INDIVIDUAL_DIR = OUTPUT_DIR / "gradcam_individual"
INDIVIDUAL_DIR.mkdir(exist_ok=True)

for i, r in enumerate(results):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))

    axes[0].imshow(r['display_img'], cmap='gray')
    axes[0].set_title(f"Original OCT Scan\nTrue Class: {r['true_class']}",
                      fontsize=11, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(r['heatmap_color'])
    axes[1].set_title("Grad-CAM Attention Map\n(bright = high model attention)",
                      fontsize=11, fontweight='bold')
    axes[1].axis('off')

    color = 'darkgreen' if r['correct'] else 'darkred'
    symbol = 'CORRECT ✓' if r['correct'] else 'INCORRECT ✗'
    axes[2].imshow(r['overlay'])
    axes[2].set_title(
        f"Heatmap Overlay\nPredicted: {r['pred_class']} ({r['confidence']:.2f}%)\n{symbol}",
        fontsize=11, fontweight='bold', color=color
    )
    axes[2].axis('off')

    plt.tight_layout()
    fname = f"sample_{i+1:02d}_{r['true_class']}_pred_{r['pred_class']}.png"
    plt.savefig(INDIVIDUAL_DIR / fname, dpi=120, bbox_inches='tight', facecolor='white')
    plt.close()

print(f"✓ Saved {len(results)} individual PNGs to {INDIVIDUAL_DIR}")

## 9 · Summary Report — For Chapter 4 of the Project Report

Prints a human-readable summary of the Grad-CAM analysis. Copy this text directly into Chapter 4 Section 4.5 (Grad-CAM Interpretability Analysis).


In [ ]:
import json

# Aggregate stats
n_total = len(results)
n_correct = sum(1 for r in results if r['correct'])
mean_conf_correct = np.mean([r['confidence'] for r in results if r['correct']]) if n_correct else 0
mean_conf_wrong = np.mean([r['confidence'] for r in results if not r['correct']]) if (n_total - n_correct) else 0

summary = {
    "grad_cam_samples": n_total,
    "correctly_classified": n_correct,
    "sample_accuracy_pct": round(100 * n_correct / n_total, 2),
    "mean_confidence_correct": round(float(mean_conf_correct), 2),
    "mean_confidence_incorrect": round(float(mean_conf_wrong), 2) if n_correct != n_total else None,
    "per_sample": [
        {
            "file": r['path'].name,
            "true_class": r['true_class'],
            "predicted_class": r['pred_class'],
            "confidence_pct": round(float(r['confidence']), 2),
            "correct": bool(r['correct']),
        }
        for r in results
    ]
}

with open(OUTPUT_DIR / "gradcam_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("GRAD-CAM ANALYSIS SUMMARY")
print("=" * 60)
print(f"Samples analyzed              : {n_total}")
print(f"Correctly classified          : {n_correct} ({100*n_correct/n_total:.1f}%)")
print(f"Mean confidence (correct)     : {mean_conf_correct:.2f}%")
if n_correct != n_total:
    print(f"Mean confidence (incorrect)   : {mean_conf_wrong:.2f}%")
print("=" * 60)
print("\nPer-sample results:")
for r in results:
    tag = "✓" if r['correct'] else "✗"
    print(f"  {tag} {r['true_class']:<7} → {r['pred_class']:<7} ({r['confidence']:5.2f}%)  [{r['path'].name}]")
print("\n✓ Summary saved to gradcam_summary.json")

## 10 · Files Generated

All outputs are in `/kaggle/working/final_results/`:

| File | Purpose |
|---|---|
| `gradcam_comparison.png` | **Main PPT slide image** — 4×3 grid, one per class |
| `gradcam_grid_full.png` | Expanded grid — 8 samples for report Chapter 4 |
| `gradcam_individual/` | 8 standalone PNGs, one per sample |
| `gradcam_summary.json` | Metrics for the interpretability section |

### What to do after this notebook completes

1. Download all files from `/kaggle/working/final_results/gradcam_*` to your laptop
2. Send them to me — I'll integrate them into the updated PPT and Chapter 4

### Interpretation guide for the PPT / report

For each Grad-CAM heatmap, look for:

- **CNV** — attention should cluster around the choroidal region (deep retinal layer disruption)
- **DME** — attention should focus on the macula, especially fluid-filled cystoid spaces
- **DRUSEN** — attention should highlight the retinal pigment epithelium region where drusen deposits appear
- **NORMAL** — attention distributed across the full retinal layer stack, without disease-specific focal points

If the heatmaps concentrate on **clinically meaningful anatomical regions**, this validates that the model has learned genuine pathological features rather than dataset artifacts — the core claim of the interpretability branch.
